# LLM Inference Engine — run everything

**Runtime → Change runtime type → T4 GPU.** Then Runtime → Run all. ~25–40 min total.

Produces: `bench/results.csv`, `bench/ppl.csv`, `bench/throughput.png`, and the filled-in resume numbers at the bottom.

In [ ]:
!nvidia-smi -L
!pip install -q transformers safetensors accelerate datasets matplotlib ninja pandas pyarrow
!git clone https://github.com/ArmaanAgarwal/llm-engine.git 2>/dev/null || (cd llm-engine && git pull)
%cd llm-engine

## 1. Correctness: our engine vs Hugging Face (must print PASS)

In [ ]:
!python -m tests.test_engine

## 2. Throughput benchmark (~15 min)

In [ ]:
!python -m bench.bench --runs 3 --new-tokens 128 --skip vllm

## 3. Quantization accuracy: perplexity fp16 vs INT8 (~3 min)

In [ ]:
!python -m bench.ppl --windows 40

## 4. Profile one decode step with Nsight Systems, before/after the fused kernel

In [ ]:
%%writefile bench/profile_step.py
import sys, torch
from engine.model import from_pretrained, KVCache
from engine import kernels
fused = sys.argv[1] == 'fused'
m = from_pretrained(device='cuda', dtype=torch.float16)
if fused: kernels.build(); kernels.USE_FUSED = True
ids = torch.randint(0, 1000, (1, 16), device='cuda')
cache = KVCache(m.cfg, 1, 64, 'cuda', torch.float16)
m.prefill(ids, cache)
tok = ids[:, -1:]
for _ in range(5): tok = m.decode(tok, cache)[:, -1].argmax(-1, keepdim=True)   # warmup
torch.cuda.synchronize()
torch.cuda.nvtx.range_push('decode_steps')
for _ in range(20): tok = m.decode(tok, cache)[:, -1].argmax(-1, keepdim=True)
torch.cuda.synchronize(); torch.cuda.nvtx.range_pop()
print('done', 'fused' if fused else 'pytorch')

In [ ]:
!nsys profile -o bench/decode_pytorch --force-overwrite true --stats=false python bench/profile_step.py pytorch 2>&1 | tail -2
!nsys profile -o bench/decode_fused   --force-overwrite true --stats=false python bench/profile_step.py fused   2>&1 | tail -2
!nsys stats --report cuda_gpu_kern_sum --format csv bench/decode_pytorch.nsys-rep 2>/dev/null | head -12
!nsys stats --report cuda_gpu_kern_sum --format csv bench/decode_fused.nsys-rep   2>/dev/null | head -12

## 5. Chart + resume numbers

In [ ]:
!python -m bench.plot
from IPython.display import Image, display; display(Image('bench/throughput.png'))
print(open('bench/ppl.csv').read())

## 6. vLLM comparison (optional, ~10 min install; skip if it fails)

In [ ]:
!pip install -q vllm 2>&1 | tail -1
!python -m bench.bench --runs 3 --new-tokens 128 --skip hf,nocache,int8,fused --out bench/results_vllm.csv

## 7. Commit results

In [ ]:
!git config user.email 'agarwal.armaan27@gmail.com' && git config user.name 'Armaan Agarwal'
!git add -A && git commit -m 'Benchmarks, perplexity, profiles' && git push